# Advanced Examples: XOR and MNIST Benchmark

In this notebook, we demonstrate the advanced capabilities of our NumPy-based neural network module, including:
1. Solving the **XOR** problem (non-linear boundary).
2. **MNIST Digits** classification using Softmax and Categorical Cross-Entropy.
3. Benchmarking against **scikit-learn**'s `MLPClassifier`.

In [ ]:
import numpy as np
import sys
import os
import matplotlib.pyplot as plt
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# Add path to import local module
sys.path.append('..')

from neural_network_numpy import NeuralNetwork
from neural_network_numpy.activations import sigmoid, relu, softmax
from neural_network_numpy.losses import log_loss, dlog_loss, categorical_cross_entropy, dcategorical_cross_entropy

## 1. XOR Logic Gate
XOR is the classic test for neural networks because it is not linearly separable.

In [ ]:
X = np.array([[0,0], [0,1], [1,0], [1,1]])
Y = np.array([[0], [1], [1], [0]])

modele_xor = NeuralNetwork(input_dim=2)
modele_xor.add_layer(4, activation=sigmoid)
modele_xor.add_layer(1, activation=sigmoid)

modele_xor.fit(X, Y, epochs=5000, lr=0.5, verbose=0, shuffle=False)

print("XOR Predictions:")
preds = modele_xor.forward(X)
for i in range(len(X)):
    print(f"Input: {X[i]} -> Target: {Y[i][0]} -> Predicted: {preds[i][0]:.4f}")

## 2. MNIST Digits Classification
We use Softmax output and Categorical Cross-Entropy to classify handwritten digits.

In [ ]:
print("Loading MNIST...")
X_mnist, y_mnist = fetch_openml('mnist_784', version=1, return_X_y=True, as_frame=False, parser='auto')
X_mnist = X_mnist[:10000] / 255.0  # Subset for speed
y_mnist = y_mnist[:10000].astype(int)

X_train, X_test, y_train, y_test = train_test_split(X_mnist, y_mnist, test_size=0.2, random_state=42)

encoder = OneHotEncoder(sparse_output=False)
y_train_cat = encoder.fit_transform(y_train.reshape(-1, 1))
y_test_cat = encoder.transform(y_test.reshape(-1, 1))

print(f"Training Data Shape: {X_train.shape}")

In [ ]:
nn = NeuralNetwork(input_dim=784)
nn.add_layer(128, activation=relu)
nn.add_layer(10, activation=softmax)

print("Training Custom NumPy Model...")
nn.fit(X_train, y_train_cat, epochs=50, lr=0.01, batch_size=128, 
       loss_fn=categorical_cross_entropy, dloss_fn=dcategorical_cross_entropy, verbose=1)

y_pred_probs = nn.forward(X_test)
y_pred = np.argmax(y_pred_probs, axis=1)
custom_accuracy = np.mean(y_pred == y_test)
print(f"Custom Model Accuracy: {custom_accuracy*100:.2f}%")

## 3. Comparison with Scikit-Learn

In [ ]:
print("Training Scikit-Learn MLPClassifier...")
mlp = MLPClassifier(hidden_layer_sizes=(128,), activation='relu', solver='sgd', 
                    learning_rate_init=0.01, max_iter=50, batch_size=128)
mlp.fit(X_train, y_train)

sklearn_accuracy = mlp.score(X_test, y_test)
print(f"Sklearn Model Accuracy: {sklearn_accuracy*100:.2f}%")

plt.figure(figsize=(10, 5))
plt.plot(nn.loss_history, label='Custom NumPy Loss')
plt.plot(mlp.loss_curve_, label='Sklearn Loss Curve')
plt.title('Loss Comparison')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend()
plt.grid(True)
plt.show()